# Garmin x Gemini - מנתח אימונים חכם

**מה הכלי עושה:**
1. מתחבר לגרמין ושולף נתוני אימונים
2. מנתח ביומכניקה של ריצה (קידוד, עוצמה, תנודה)
3. שולח לGemini AI לניתוח מקצועי בעברית
4. יוצר תוכנית אימון שבועית
5. מעלה אימונים לגרמין (יגיעו לשעון)
6. שומר נתונים באתר האישי

---
**לפני שמתחילים צריך:**
- אימייל וסיסמה של Garmin Connect
- מפתח Gemini API חינמי מ: https://aistudio.google.com/apikey
- GitHub Token מ: https://github.com/settings/tokens (בחר repo)

**איך מריצים:** מלא פרטים בתא הבא -> לחץ Runtime -> Run all

## שלב 1: התקנת ספריות

In [ ]:
!pip install garminconnect google-genai fitparse requests --quiet
print('ספריות הותקנו!')

## שלב 2: הגדרות - ערוך את השורות הבאות!

In [ ]:
# ========== ערוך את השורות הבאות ==========
GARMIN_EMAIL    = "your@email.com"
GARMIN_PASSWORD = "your-password"
GEMINI_API_KEY  = "your-gemini-api-key"
GITHUB_TOKEN    = "ghp_your_token"
GITHUB_REPO     = "Hadar2255/Hadar2255"
DAYS_BACK       = 14
MAX_ACTIVITIES  = 10
# ============================================
print('הגדרות נשמרו!')

## שלב 3: שליפת נתוני Garmin

In [ ]:
import garminconnect
from datetime import date, timedelta
import json, io
try:
    from fitparse import FitFile
    HAS_FITPARSE = True
except ImportError:
    HAS_FITPARSE = False

print('מתחבר ל-Garmin Connect...')
garmin = garminconnect.Garmin(GARMIN_EMAIL, GARMIN_PASSWORD)
garmin.login()
print('מחובר!')

today      = date.today()
start_date = today - timedelta(days=DAYS_BACK)
last_7     = [today - timedelta(days=i) for i in range(7)]

print(f'שולף פעילויות...')
try:
    activities = garmin.get_activities_by_date(str(start_date), str(today), 'running')
except Exception:
    activities = garmin.get_activities(0, 20)
print(f'נמצאו {len(activities)} ריצות')

sleep_data = []
for d in last_7:
    try:
        s = garmin.get_sleep_data(str(d))
        if s:
            daily = s.get('dailySleepDTO', {})
            sleep_data.append({
                'date': str(d),
                'duration_hours': round(daily.get('sleepTimeSeconds', 0) / 3600, 1),
                'deep_sleep_hours': round(daily.get('deepSleepSeconds', 0) / 3600, 1),
                'rem_sleep_hours': round(daily.get('remSleepSeconds', 0) / 3600, 1),
                'score': daily.get('sleepScores', {}).get('overall', {}).get('value', 'N/A')
            })
    except Exception:
        pass

hrv_data = []
for d in last_7:
    try:
        h = garmin.get_hrv_data(str(d))
        if h and h.get('hrvSummary'):
            hrv_data.append({
                'date': str(d),
                'weekly_avg': h['hrvSummary'].get('weeklyAvg'),
                'last_night': h['hrvSummary'].get('lastNight'),
                'status': h['hrvSummary'].get('status')
            })
    except Exception:
        pass

biomechanics_list = []
for activity in activities[:MAX_ACTIVITIES]:
    act_id = activity.get('activityId')
    bio = {
        'date': activity.get('startTimeLocal', '')[:10],
        'name': activity.get('activityName', 'ריצה'),
        'distance_km': round(activity.get('distance', 0) / 1000, 2),
        'duration_min': round(activity.get('duration', 0) / 60, 1),
        'avg_hr': activity.get('averageHR'),
        'max_hr': activity.get('maxHR'),
        'avg_cadence': activity.get('averageRunningCadenceInStepsPerMinute'),
        'avg_power': activity.get('avgPower'),
        'normalized_power': activity.get('normPower'),
        'aerobic_te': activity.get('aerobicTrainingEffect'),
        'anaerobic_te': activity.get('anaerobicTrainingEffect'),
        'training_load': activity.get('activityTrainingLoad'),
        'training_effect_label': activity.get('trainingEffectLabel'),
        'avg_speed_kmh': round(activity.get('averageSpeed', 0) * 3.6, 2) if activity.get('averageSpeed') else None,
        'avg_gct_ms': None,
        'avg_vertical_oscillation_cm': None,
        'avg_vertical_ratio': None,
        'avg_stride_length_m': None,
    }
    if HAS_FITPARSE and act_id:
        try:
            fit_bytes = garmin.download_activity(act_id, dl_fmt=garminconnect.Garmin.ActivityDownloadFormat.ORIGINAL)
            fitfile = FitFile(io.BytesIO(fit_bytes))
            gct_v, vo_v, vr_v, sl_v = [], [], [], []
            for record in fitfile.get_messages('record'):
                data = {f.name: f.value for f in record}
                if data.get('stance_time'):          gct_v.append(data['stance_time'])
                if data.get('vertical_oscillation'): vo_v.append(data['vertical_oscillation'])
                if data.get('vertical_ratio'):       vr_v.append(data['vertical_ratio'])
                if data.get('step_length'):          sl_v.append(data['step_length'])
            if gct_v: bio['avg_gct_ms'] = round(sum(gct_v)/len(gct_v), 1)
            if vo_v:  bio['avg_vertical_oscillation_cm'] = round(sum(vo_v)/len(vo_v)/10, 1)
            if vr_v:  bio['avg_vertical_ratio'] = round(sum(vr_v)/len(vr_v), 1)
            if sl_v:  bio['avg_stride_length_m'] = round(sum(sl_v)/len(sl_v)/1000, 2)
        except Exception:
            pass
    biomechanics_list.append(bio)

cadence_vals = [b['avg_cadence'] for b in biomechanics_list if b['avg_cadence']]
avg_cadence  = round(sum(cadence_vals)/len(cadence_vals), 1) if cadence_vals else 'N/A'
gct_vals     = [b['avg_gct_ms'] for b in biomechanics_list if b['avg_gct_ms']]
avg_gct      = round(sum(gct_vals)/len(gct_vals), 1) if gct_vals else 'N/A'
vo_vals      = [b['avg_vertical_oscillation_cm'] for b in biomechanics_list if b['avg_vertical_oscillation_cm']]
avg_vert_osc = round(sum(vo_vals)/len(vo_vals), 1) if vo_vals else 'N/A'
print(f'ריצות: {len(activities)} | קידוד ממוצע: {avg_cadence} spm | GCT: {avg_gct} ms | תנודה: {avg_vert_osc} cm')

## שלב 4: ניתוח Gemini AI

In [ ]:
import google.genai as genai
client = genai.Client(api_key=GEMINI_API_KEY)

acts_text = ''
for b in biomechanics_list:
    acts_text += f"""
  {b['date']} - {b['name']}
  מרחק: {b['distance_km']} קמ | זמן: {b['duration_min']} דק | קצב: {b['avg_speed_kmh']} קמש
  דופק: {b['avg_hr']} | קידוד: {b['avg_cadence']} spm | עוצמה: {b['avg_power']} W
  Training Effect: אירובי {b['aerobic_te']} אנאירובי {b['anaerobic_te']} ({b['training_effect_label']})
  GCT: {b['avg_gct_ms']} ms | תנודה: {b['avg_vertical_oscillation_cm']} cm"""

sleep_text = '\n'.join([f"  {s['date']}: {s['duration_hours']}h (עמוק:{s['deep_sleep_hours']}h ציון:{s['score']})" for s in sleep_data])
hrv_text   = '\n'.join([f"  {h['date']}: {h['last_night']} ms ({h['status']})" for h in hrv_data])

prompt = f"""אתה מאמן ריצה מקצועי ומומחה ביומכניקה. נתוני הספורטאי ב-{DAYS_BACK} הימים האחרונים:

פעילויות:
{acts_text}

שינה: {sleep_text or 'לא זמין'}
HRV:  {hrv_text or 'לא זמין'}

ממוצעי ביומכניקה:
  קידוד: {avg_cadence} spm (מטרה 170-180)
  GCT:   {avg_gct} ms (מטרה 200-270)
  תנודה: {avg_vert_osc} cm (מטרה 6-9)

ספק בעברית:
## 1. ניתוח ביומכניקה - קידוד, GCT, תנודה אנכית
## 2. ניתוח עומס אימון והתאוששות לפי HRV ושינה
## 3. תוכנית שבועית (3-5 ריצות) עם: סוג/מרחק/דופק/קצב מטרה
## 4. תרגילי שיפור ביומכניקה
היה ספציפי עם מספרים."""

print('שולח לGemini...')
response = client.models.generate_content(model='gemini-2.0-flash', contents=prompt)
gemini_analysis = response.text
print('ניתוח התקבל!')

## שלב 5: הניתוח שלך

In [ ]:
from IPython.display import Markdown, display
display(Markdown(gemini_analysis))

## שלב 6: יצירת אימונים בגרמין

In [ ]:
import re

workout_prompt = f"""בהתבסס על הניתוח:
{gemini_analysis}

הפק JSON בלבד (ללא הסברים) עם מערך אימונים:
[{{"name":"שם","type":"easy|tempo|intervals|long","day_offset":1,
  "warmup_minutes":10,"main_minutes":30,"cooldown_minutes":10,
  "target_hr_zone":2,"description":"תיאור"}}]
day_offset: 1=מחר. עד 5 אימונים."""

wr = client.models.generate_content(model='gemini-2.0-flash', contents=workout_prompt)
jm = re.search(r'\\[.*?\\]', wr.text, re.DOTALL)
planned = []
if jm:
    try: planned = json.loads(jm.group())
    except: pass

if not planned:
    planned = [
        {'name':'ריצה קלה','type':'easy','day_offset':1,'warmup_minutes':5,'main_minutes':25,'cooldown_minutes':5,'target_hr_zone':2,'description':'שחרור'},
        {'name':'ריצת טמפו','type':'tempo','day_offset':3,'warmup_minutes':10,'main_minutes':20,'cooldown_minutes':10,'target_hr_zone':4,'description':'קצב תחרותי'},
        {'name':'ריצה ארוכה','type':'long','day_offset':6,'warmup_minutes':10,'main_minutes':50,'cooldown_minutes':10,'target_hr_zone':2,'description':'ריצה ארוכה'}
    ]

created_workouts = []
hr_targets = {1:(100,120),2:(120,140),3:(140,155),4:(155,168),5:(168,185)}

for w in planned:
    workout_date = today + timedelta(days=w.get('day_offset',1))
    hr_min, hr_max = hr_targets.get(w.get('target_hr_zone',2),(120,140))
    payload = {
        'sportType':{'sportTypeId':1,'sportTypeKey':'running'},
        'workoutName': w['name'],
        'description': w.get('description',''),
        'workoutSegments':[{'segmentOrder':1,
            'sportType':{'sportTypeId':1,'sportTypeKey':'running'},
            'workoutSteps':[
                {'stepOrder':1,'stepType':{'stepTypeId':1,'stepTypeKey':'warmup'},
                 'endCondition':{'conditionTypeId':2,'conditionTypeKey':'time'},
                 'endConditionValue':w.get('warmup_minutes',10)*60,
                 'targetType':{'workoutTargetTypeId':4,'workoutTargetTypeKey':'heart.rate.zone'},
                 'targetValueOne':100,'targetValueTwo':130},
                {'stepOrder':2,'stepType':{'stepTypeId':3,'stepTypeKey':'interval'},
                 'endCondition':{'conditionTypeId':2,'conditionTypeKey':'time'},
                 'endConditionValue':w.get('main_minutes',30)*60,
                 'targetType':{'workoutTargetTypeId':4,'workoutTargetTypeKey':'heart.rate.zone'},
                 'targetValueOne':hr_min,'targetValueTwo':hr_max},
                {'stepOrder':3,'stepType':{'stepTypeId':2,'stepTypeKey':'cooldown'},
                 'endCondition':{'conditionTypeId':2,'conditionTypeKey':'time'},
                 'endConditionValue':w.get('cooldown_minutes',10)*60,
                 'targetType':{'workoutTargetTypeId':4,'workoutTargetTypeKey':'heart.rate.zone'},
                 'targetValueOne':100,'targetValueTwo':130}
            ]}]
    }
    try:
        result = garmin.add_workout(payload)
        wid = result.get('workoutId')
        if wid: garmin.schedule_workout(wid, str(workout_date))
        created_workouts.append({'name':w['name'],'date':str(workout_date),'id':wid})
        print(f'  נוצר: {w["name"]} ל-{workout_date}')
    except Exception as e:
        print(f'  שגיאה ב-{w["name"]}: {e}')

print(f'\n{len(created_workouts)} אימונים נוצרו ב-Garmin Connect!')
print('סנכרן את השעון לאפליקציה והם יופיעו!')

## שלב 7: שמירת ניתוח באתר

In [ ]:
import requests, base64

GITHUB_API = 'https://api.github.com'
headers = {
    'Authorization': f'Bearer {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28'
}

def push_json(filename, data):
    content_b64 = base64.b64encode(json.dumps(data, ensure_ascii=False, indent=2).encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{filename}'
    sha = None
    r = requests.get(url, headers=headers)
    if r.status_code == 200: sha = r.json().get('sha')
    payload = {'message': f'ניתוח {today}', 'content': content_b64, 'branch': 'main'}
    if sha: payload['sha'] = sha
    r = requests.put(url, headers=headers, json=payload)
    return r.status_code in (200, 201)

record = {
    'date': str(today),
    'summary': {'total_runs': len(activities), 'avg_cadence': avg_cadence,
                'avg_gct_ms': avg_gct, 'avg_vertical_oscillation_cm': avg_vert_osc},
    'activities': biomechanics_list,
    'sleep': sleep_data,
    'hrv': hrv_data,
    'gemini_analysis': gemini_analysis,
    'created_workouts': created_workouts
}

push_json(f'data/{today}.json', record)
push_json('data/latest.json', record)

r = requests.get(f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/data/index.json', headers=headers)
idx = json.loads(base64.b64decode(r.json()['content']).decode()) if r.status_code==200 else []
idx = [e for e in idx if e.get('date') != str(today)]
idx.append({'date':str(today),'file':f'data/{today}.json','runs':len(activities),'avg_cadence':avg_cadence})
idx.sort(key=lambda x: x['date'], reverse=True)
push_json('data/index.json', idx)

print('נשמר ב-GitHub!')
print(f'האתר שלך: https://hadar2255.github.io/Hadar2255')
print(f'נוצרו {len(created_workouts)} אימונים | נותחו {len(activities)} ריצות')